# SE(3)-Transformer Overview
The SE(3)-Transformer is a Graph Neural Network using a variant of self-attention for 3D points and graphs processing.
This model is equivariant under continuous 3D roto-translations, meaning that when the inputs (graphs or sets of points) rotate in 3D space
(or more generally experience a proper rigid transformation), the model outputs either stay invariant or transform with the input.

These imports set up the full SE(3)-Transformer training and evaluation pipeline on the QM9 molecular dataset — covering data loading, distributed training, optimization, logging, and inference

In [1]:
import logging

import torch.nn as nn
import dgl

from se3_transformer.data_loading import QM9DataModule
from se3_transformer.model import SE3TransformerPooled
from se3_transformer.model.fiber import Fiber
from se3_transformer.runtime.arguments import PARSER
from se3_transformer.runtime.callbacks import (
    QM9MetricCallback,
    QM9LRSchedulerCallback,
)
from se3_transformer.runtime.loggers import (
    LoggerCollection,
    DLLogger,
)
from se3_transformer.runtime.utils import (
    seed_everything,
    using_tensor_cores,
)
from se3_transformer.runtime.training import train

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


/opt/venv/lib/python3.10/site-packages/apex/transformer/functional/fused_rope.py:54: UserWarning: Using the native apex kernel for RoPE.
  warnings.warn("Using the native apex kernel for RoPE.", UserWarning)


## Using the CLI's args to setup training
Using CLI Arguments to Set Up Training

The SE(3)-Transformer example from the Deep Learning Examples repository was originally designed to run as a command-line program — but we can easily adapt it for use in Jupyter notebooks! The training configuration, including model, optimizer, and runtime settings, is managed through an argparse parser, which we can leverage directly within the notebook for flexible experimentation.

In [2]:
# Uncomment the line below to see all available training and runtime arguments
PARSER.print_help()

# Adjust the following parameters as needed for your system configuration.
args = PARSER.parse_args(
    [
        "--epochs",
        "5",
        "--eval_interval",
        "1",
        "--batch_size",
        "240",
        "--num_workers",
        "16",
        "--precompute_bases",
        "--use_layer_norm",
        "--norm",
        "--save_ckpt_pat",
        "model_qm9.pth",
        # If you want to load a model trained for 100 epochs, uncomment the line below
        # "--load_ckpt_path",
        # "model_qm9_100.pth",
    ]
)
# Uncomment to verify that the args have been set properly
print(args)

usage: ipykernel_launcher.py [-h] [--data_dir DATA_DIR] [--log_dir LOG_DIR]
                             [--dllogger_name DLLOGGER_NAME]
                             [--save_ckpt_path SAVE_CKPT_PATH]
                             [--load_ckpt_path LOAD_CKPT_PATH]
                             [--optimizer {adam,sgd,lamb}]
                             [--learning_rate LEARNING_RATE]
                             [--min_learning_rate MIN_LEARNING_RATE]
                             [--momentum MOMENTUM]
                             [--weight_decay WEIGHT_DECAY] [--epochs EPOCHS]
                             [--batch_size BATCH_SIZE] [--seed SEED]
                             [--num_workers NUM_WORKERS] [--amp [AMP]]
                             [--gradient_clip GRADIENT_CLIP]
                             [--accumulate_grad_batches ACCUMULATE_GRAD_BATCHES]
                             [--ckpt_interval CKPT_INTERVAL]
                             [--eval_interval EVAL_INTERVAL]
                

## Dataset and Model Setup
We start by loading the QM9 molecular dataset using the QM9DataModule, which handles data preprocessing, batching, and splitting for training and evaluation.
Next, we initialize the SE(3)-Transformer model (SE3TransformerPooled) with input, edge, and output fibers that define how geometric and feature information flow through the network.
Finally, we define the L1 loss (nn.L1Loss) — a simple yet effective choice for molecular property regression tasks.

In [3]:
datamodule = QM9DataModule(**vars(args))
model = SE3TransformerPooled(
    fiber_in=Fiber({0: datamodule.NODE_FEATURE_DIM}),
    fiber_out=Fiber({0: args.num_degrees * args.num_channels}),
    fiber_edge=Fiber({0: datamodule.EDGE_FEATURE_DIM}),
    output_dim=1,
    tensor_cores=using_tensor_cores(args.amp),  # use Tensor Cores more effectively
    **vars(args),
)
loss_fn = nn.L1Loss()

Done loading data from cached files.


Precomputing QM9 bases: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 546/546 [00:50<00:00, 10.73it/s]


# ⌬ Inspecting the Molecules
Before diving into training, it’s helpful to visually inspect the molecules from the QM9 dataset.
We use RDKit to reconstruct 3D molecular structures from the graph data (node positions and atomic numbers) and py3Dmol for interactive visualization right inside the notebook.

The convert_to_mol() function converts DGL graphs into RDKit Mol objects by building a representation and determining the bonds. Then, using an interactive widget, we can scroll through the validation set and view each molecule in 3D — a great way to sanity-check data preprocessing and bonding structure.

Note : You might occasionally see RDKit throw a ValueError like
“Valence of atom X is larger than the allowed maximum”.
This happens when bond inference from raw coordinates produces chemically invalid structures.
It’s expected for a few molecules in QM9, since not all atomic configurations map perfectly back to valid 3D molecules. You can safely ignore these errors or skip those samples — they don’t affect the rest of the dataset or training. 

In [6]:
# Note: RDKit may raise a "Valence of atom ... is larger than allowed" error.
# This occurs when the inferred bonds don't form a valid molecule — it's expected for a few QM9 samples.

from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
import py3Dmol
from ipywidgets import interact, IntSlider


def convert_to_mol(graph: dgl.graph) -> Chem.Mol:
    ptable = Chem.GetPeriodicTable()

    # extract positions and atomic numbers
    raw_coords = graph.ndata["pos"]
    raw_atomic_numbers = graph.ndata["attr"][:, 5]
    n_atoms = raw_atomic_numbers.shape[0]

    # construct xyz
    xyz_str = f"{n_atoms}\n\n"
    for an, coords in zip(raw_atomic_numbers, raw_coords):
        symb = ptable.GetElementSymbol(int(an))
        xyz_str += f"{symb}    {coords[0]}    {coords[1]}    {coords[2]}\n"
    mol = Chem.MolFromXYZBlock(xyz_str)

    # get bonds, and go from 2D ->
    rdDetermineBonds.DetermineBonds(mol)
    return mol


# Get the dataloader for the validation dataset
val_loader = datamodule.val_dataloader()


# Define a function that takes the index as a parameter
def visualize_molecule(index):
    try:
        mol = convert_to_mol(val_loader.dataset[0][0])
        mb = Chem.MolToMolBlock(mol)
        # Add your visualization code here
        # For example, if you're using py3Dmol:
        view = py3Dmol.view(width=400, height=400)
        view.addModel(mb, "sdf")
        view.setStyle({"stick": {}})
        view.zoomTo()
        return view.show()
    except ValueError as e:
        print(f"We cannot visualize this molecule: {e}")


# Create the interactive widget
interact(
    visualize_molecule,
    index=IntSlider(
        min=0,
        max=len(val_loader.dataset) - 1,
        step=1,
        value=18,
        description="Validation Index:",
    ),
)

interactive(children=(IntSlider(value=18, description='Validation Index:', max=17747), Output()), _dom_classes…

<function __main__.visualize_molecule(index)>

# Input Representation for a Single Molecule

Let us look at one of the molecules in the dataset

![molecule](molecule.png)

## Basic Graph Information

``` text
--- BASIC INFO ---
Nodes: 14
Edges: 28
```

The molecule is represented as a graph with 14 nodes corresponding to atoms and 28 edges representing atom–atom interactions.
Edges are constructed based on interatomic proximity rather than explicit chemical bonds.

## Node Features (ndata)

Each node (atom) is associated with geometric and chemical features.

### Atomic Positions

``` text
Key: pos
Shape: (14, 3)
Dtype: torch.float32

tensor([[ 0.6781, -0.0583,  0.7324],
        [-0.1432,  0.3297, -0.3889],
        [-1.5010, -0.2886, -0.2937],
        ...])

```
Each row represents the 3D Cartesian coordinates [x,y,z] of an atom in the molecule.

### Atomic Attributes

``` text
Key: attr
Shape: (14, 11)
Dtype: torch.float32

tensor([[0., 0., 0., 1., 0., 8., 0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 2.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 0.],
        ...])
```
Each row encodes atom-specific properties, such as atomic type and related categorical or numerical descriptors, which allow the model to distinguish between different elements.

## Edge Features (edata)

Edges capture pairwise relationships between atoms.

``` text
Edge Attributes
Key: edge_attr
Shape: (28, 4)
Dtype: torch.float32

tensor([[1., 0., 0., 0.],
        [1., 0., 0., 0.],
        [1., 0., 0., 0.],
        ...])
```
Each row represents a feature vector associated with an edge, typically encoding distance-based or radial information used to model interatomic interactions.

## Regression Target
```text
--- TARGET FOR HOMO  (HIGHEST UNOCCUPIED MOLECULAR ORBITAL ENERGY)  ---
Shape: torch.Size([1])
Value: tensor([-5.7987])
```

The regression target corresponds to the HOMO (Highest Occupied Molecular Orbital) energy of the molecule, expressed in electron volts (eV).
This value represents a global molecular property and serves as the supervision signal during training.

Similarly for other regresion targets -
```text
--- TARGET FOR LUMO (LOWEST UNOCCUPIED MOLECULAR ORBITAL ENERGY) ---
Shape: torch.Size([1])
Value: tensor([0.9905])
```
```text
--- TARGET FOR GAP (Gap between HOMO and LUMO) ---
Shape: torch.Size([1])
Value: tensor([6.7892])
```

# Model Summary
We can quickly inspect the SE(3)-Transformer architecture using torchinfo.summary, which prints a detailed overview of each layer, its input/output shapes, and the number of parameters. This helps us verify that the model is built correctly before training.

In [ ]:
from torchinfo import summary

summary(model)

# Logging and callbacks
Before training, we set up logging, seeding, and callbacks to keep the experiment organized and reproducible. The logging level is set to INFO so key messages about configuration and progress are visible. If a random seed is provided, it is initialized to ensure reproducibility across runs. We create a DLLogger (wrapped in a LoggerCollection) to save logs, and configure callbacks like QM9MetricCallback for validation metrics and QM9LRSchedulerCallback for learning rate scheduling. Finally, all hyperparameters from args are recorded in the logger to track and reproduce experiments consistently.

In [ ]:
# Initialize logging, set seed, configure loggers and training callbacks
logging.getLogger().setLevel(logging.INFO)

if args.seed is not None:
    logging.info(f"Using seed {args.seed}")
    seed_everything(args.seed)

logging.info(f"Saving info to {args.log_dir}/{args.dllogger_name}")
loggers = [DLLogger(save_dir=args.log_dir, filename=args.dllogger_name)]
logger = LoggerCollection(loggers)
callbacks = [
    QM9MetricCallback(logger, targets_std=datamodule.targets_std, prefix="validation"),
    QM9LRSchedulerCallback(logger, epochs=args.epochs),
]
logger.log_hyperparams(vars(args))

# Train
With everything configured, we’re ready to kick off training. The train() function orchestrates the entire training loop — running forward and backward passes, computing losses, updating parameters, and periodically evaluating on the validation set. It uses the dataloaders, callbacks, and logger we set up earlier to track progress, log metrics, and manage learning rate schedules throughout the training process.

In [ ]:
train(
    model,
    loss_fn,
    datamodule.train_dataloader(),
    datamodule.val_dataloader(),
    callbacks,
    logger,
    args,
)

# Visualizing Training Progress
After training, we can visualize and analyze the logged results. We import Plotly for interactive plotting and dllogger to access the saved training logs. Flushing the logger ensures all metrics have been written to disk before loading them.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
from plotly.subplots import make_subplots
import json
import dllogger
import os

# If we're loading a checkpoint, we need to use the saved log file
# otherwise, we'll use the current log file
if args.load_ckpt_path is not None:
    LOG_FILE = os.path.join("results", "dllogger_results_100.json")
    if not os.path.exists(LOG_FILE):
        raise FileNotFoundError(f"Log file {LOG_FILE} does not exist, please copy the log file to the results directory or turn off checkpoint loading")
else:
    LOG_FILE = os.path.join("results", args.log_dir, args.dllogger_name)
    dllogger.flush()

print(f"Using log file: {LOG_FILE}")
pio.renderers.default = "notebook"

This step parses and organizes the logged training data from dllogger_results.json. We read the file line by line, clean up any malformed entries, and filter out records without valid steps. Each log entry is then grouped by its training step, extracting key metrics such as training loss, learning rate, and validation MAE. The results are compiled into a tidy Pandas DataFrame, making it easier to visualize and analyze how model performance and learning dynamics evolved throughout training.

In [ ]:
# Read and parse the data
with open(LOG_FILE, "r") as f:
    logs = [json.loads(line.replace("DLLL", "")) for line in f.readlines()]

# Filter out entries where step is an empty list
logs = [log for log in logs if log.get("step") != []]

# Create a dictionary to aggregate metrics by step
metrics_by_step = {}

for log in logs:
    if log.get("type") == "LOG":
        step = log.get("step")

        # Skip if step is not an integer or if it's the PARAMETER step
        if not isinstance(step, int):
            continue

        # Initialize the step if not exists
        if step not in metrics_by_step:
            metrics_by_step[step] = {
                "step": step,
                "train loss": None,
                "learning rate": None,
                "validation MAE": None,
            }

        # Update metrics for this step
        data = log.get("data", {})
        if "train loss" in data:
            metrics_by_step[step]["train loss"] = data["train loss"]
        if "learning rate" in data:
            metrics_by_step[step]["learning rate"] = data["learning rate"]
        if "validation MAE" in data:
            metrics_by_step[step]["validation MAE"] = data["validation MAE"]

# Convert to DataFrame
df = pd.DataFrame(list(metrics_by_step.values()))
df = df.sort_values("step").reset_index(drop=True)

print(df)

To get a clear picture of how training evolved, we plot the key metrics over epochs using Plotly. The figure below displays training loss, validation MAE, and learning rate in separate subplots, making it easy to observe the model’s convergence and learning dynamics. Ideally, you should see the training loss and validation MAE steadily decreasing as the learning rate adjusts — giving a quick visual confirmation that training progressed smoothly.

In [ ]:
# Create subplots
fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=("Train Loss", "Validation MAE", "Learning Rate"),
    vertical_spacing=0.08,
)

# Train Loss
fig.add_trace(
    go.Scatter(
        x=df["step"],
        y=df["train loss"],
        mode="lines+markers",
        name="Train Loss",
        line=dict(color="blue"),
    ),
    row=1,
    col=1,
)

# Validation MAE
fig.add_trace(
    go.Scatter(
        x=df["step"],
        y=df["validation MAE"],
        mode="lines+markers",
        name="Validation MAE",
        line=dict(color="red"),
    ),
    row=2,
    col=1,
)

# Learning Rate
fig.add_trace(
    go.Scatter(
        x=df["step"],
        y=df["learning rate"],
        mode="lines+markers",
        name="Learning Rate",
        line=dict(color="green"),
    ),
    row=3,
    col=1,
)

fig.update_xaxes(title_text="Epoch", row=3, col=1)
fig.update_layout(height=1000, showlegend=False, title_text="SE(3) Training")
fig.show()

# Inference

After training, we now look at inference and how it is handled by DGL.

In [ ]:
predict()

# Conclusion

In this notebook, we walked through the end-to-end workflow for training and evaluating an SE(3)-Transformer model on the QM9 molecular dataset. We explored how to set up training configurations originally designed for CLI use, adapted them for an interactive Jupyter workflow, and visualized molecules directly from graph data to validate preprocessing. We then built and trained the SE(3)-Transformer, logged its performance, and used interactive plots to analyze key metrics like loss, MAE, and learning rate over time.

With the workflow now validated, this setup provides a strong foundation for scaling up experiments, benchmarking performance, and adapting the SE(3)-Transformer to more complex or domain-specific datasets.